<div style="background:linear-gradient(135deg,#061A33 0%,#0B2E59 45%,#00A6C0 100%); padding:44px 36px; border-radius:18px; color:#F4FBFF; font-family:'Segoe UI',Tahoma,sans-serif; box-shadow:0 8px 30px rgba(0,0,0,0.25);">
  <div style="font-size:12px; letter-spacing:4px; text-transform:uppercase; opacity:0.75; margin-bottom:10px;">BinX Tech &nbsp;•&nbsp; AI &amp; Machine Learning Internship &nbsp;•&nbsp; Phase 3 Capstone</div>
  <div style="font-size:38px; font-weight:800; line-height:1.15;">🩺 Week 8 · Day 2</div>
  <div style="font-size:22px; font-weight:400; opacity:0.92; margin-top:4px;">Text Representation — TF-IDF &amp; Word Embeddings</div>
  <div style="margin-top:20px; display:flex; flex-wrap:wrap; gap:8px;">
    <span style="background:#00C2D1; color:#04263A; padding:5px 14px; border-radius:20px; font-size:12px; font-weight:700;">🚀 SPRINT 3 — DAY 2 / 5</span>
    <span style="background:#FFC857; color:#4A3400; padding:5px 14px; border-radius:20px; font-size:12px; font-weight:700;">⏱️ 8 HOURS</span>
    <span style="background:#8CE99A; color:#0B3D1E; padding:5px 14px; border-radius:20px; font-size:12px; font-weight:700;">🔢 TEXT REPRESENTATION</span>
    <span style="background:#D0BFFF; color:#2E1065; padding:5px 14px; border-radius:20px; font-size:12px; font-weight:700;">❤️ CARDIAC MONITORING THREAD</span>
  </div>
</div>

<div style="text-align:center; color:#7C8B9B; font-size:13px; font-style:italic; margin-top:14px;">
"A model doesn't read words — it reads geometry. Today we build the map."
</div>

<div style="border-left:5px solid #00A6C0; background:#F0FBFD; padding:18px 22px; border-radius:0 10px 10px 0; margin:18px 0;">
<h3 style="margin-top:0; color:#0B2E59;">📖 The Story So Far</h3>
<p style="margin-bottom:0; color:#233; line-height:1.6;">
Day 1 ended with clean, correctly-negated tokens — <code>"medication"</code>, <code>"not"</code>, <code>"help"</code>,
<code>"symptom"</code> — instead of raw, messy sentences. That's progress, but a model still can't do anything with a
list of words. It needs <b>numbers</b>. Today turns clean tokens into numeric vectors two different ways — TF-IDF and
word embeddings — and asks which one actually deserves a place in a real pipeline.
</p>
</div>

<div style="background:#0B2E59; border-radius:14px; padding:22px 26px; color:#EAF6FA;">
<h3 style="margin-top:0; color:#8CE9F4;">🎯 Learning Objectives</h3>
<table style="width:100%; border-collapse:collapse; color:#EAF6FA;">
<tr><td style="padding:8px 10px; font-size:20px;">🔢</td><td style="padding:8px 10px;">Convert cleaned text to numeric vectors with TF-IDF.</td></tr>
<tr><td style="padding:8px 10px; font-size:20px;">🧭</td><td style="padding:8px 10px;">Explain word embeddings and how they capture meaning as geometry.</td></tr>
<tr><td style="padding:8px 10px; font-size:20px;">⚖️</td><td style="padding:8px 10px;">Choose between TF-IDF and embeddings for a given task — and justify it.</td></tr>
</table>
</div>

## 2.1 &nbsp;From Text to Numbers

After cleaning, text must become numeric vectors. There are two main families of approaches — and understanding the
difference between them is central to applied NLP. Before building either one, we reload the same cleaning pipeline
from Day 1 (this folder ships its own copy of the pipeline and data so it runs standalone).

In [1]:
# --- Setup: same environment and cleaning pipeline as Day 1 ---
import nltk, re, string
import pandas as pd

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize(token):
    v = lemmatizer.lemmatize(token, pos='v')
    return v if v != token else lemmatizer.lemmatize(token, pos='n')

stop_words_default = set(stopwords.words('english'))
negation_words = {
    "not", "no", "nor", "never",
    "don", "didn", "doesn", "isn", "wasn", "won", "cant", "cannot", "couldnt", "shouldnt", "wouldnt",
}
stop_words_task_aware = stop_words_default - negation_words

def clean_task_aware(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words_task_aware]
    tokens = [lemmatize(t) for t in tokens]
    return tokens

print("Day 1 pipeline reloaded ✅")

Day 1 pipeline reloaded ✅


In [2]:
import os

CSV_PATH = "day2_sample_reviews.csv"

# Colab runs in a fresh, empty session each time — a file sitting next to this
# notebook on GitHub is NOT automatically there. If it's missing, ask for it.
if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        print(f"'{CSV_PATH}' isn't in this Colab session yet. Upload it now (the file you downloaded earlier):")
        files.upload()
    except ImportError:
        pass  # not running in Colab — fall through to the fallback below

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
else:
    # last-resort fallback so this cell never hard-crashes even without the file
    df = pd.DataFrame({
        "review_id": [1, 2, 3, 4],
        "review": [
            "This medication did NOT help my symptoms at all, I was very disappointed.",
            "Great drug, no side effects, and it really helped with my anxiety.",
            "I stopped taking it because the side effects were too strong for me.",
            "Amazing improvement in my symptoms after just one week on this medication.",
        ],
        "sentiment": ["negative", "positive", "negative", "positive"],
    })
    print(f"'{CSV_PATH}' not found — using a tiny 4-row built-in fallback instead of the full 20-row file.")

df["cleaned"] = df["review"].apply(lambda t: " ".join(clean_task_aware(t)))
print(f"Loaded {len(df)} labeled reviews.\n")
df[["review", "sentiment", "cleaned"]].head()

Loaded 20 labeled reviews.



,review,sentiment,cleaned
0,This medication did NOT help my symptoms at al...,negative,medication not help symptom disappoint
1,"""Amazing results!"" My doctor said it's the bes...",positive,amaze result doctor say best option condition
2,I've been taking this for 3 months and it hasn...,negative,ive take 3 month hasnt work well
3,"Great drug, no side effects, and it really hel...",positive,great drug no side effect really help anxiety
4,The pills tasted awful and I felt worse after ...,negative,pill taste awful felt worse first week


## 2.2 &nbsp;Bag-of-Words and TF-IDF

The bag-of-words approach represents a document by which words it contains and how often, ignoring order. **TF-IDF**
(Term Frequency–Inverse Document Frequency) improves on raw counts by weighting each word: a word scores high if it's
frequent in *this* document (TF) but rare *across all* documents (IDF). This down-weights common words like "the" or
"medication" (appears everywhere here) and up-weights distinctive ones — surfacing the words that actually
distinguish one review from another.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=200)
X_tfidf = vectorizer.fit_transform(df["cleaned"])

print("TF-IDF matrix shape:", X_tfidf.shape, " (reviews × vocabulary)")

# Top-weighted terms for one specific review
row = 0
scores = X_tfidf[row].toarray().flatten()
terms = vectorizer.get_feature_names_out()
top_idx = scores.argsort()[::-1][:5]

print(f"\nReview: \"{df.loc[row, 'review']}\"")
print("Top TF-IDF terms:", [(terms[i], round(scores[i], 3)) for i in top_idx if scores[i] > 0])

TF-IDF matrix shape: (20, 63)  (reviews × vocabulary)

Review: "This medication did NOT help my symptoms at all, I was very disappointed."
Top TF-IDF terms: [('disappoint', np.float64(0.567)), ('not', np.float64(0.469)), ('symptom', np.float64(0.434)), ('medication', np.float64(0.378)), ('help', np.float64(0.356))]


## 2.3 &nbsp;Word Embeddings

Word embeddings represent each word as a dense vector positioned so that words with similar *meanings* sit close
together in vector space. Unlike TF-IDF, embeddings capture semantic relationships — famously, `king - man + woman ≈
queen`, because the geometry itself encodes meaning. Word2Vec and GloVe are the classic pre-trained embedding
methods. We load real pre-trained GloVe vectors below (50-dimensional, trained on Wikipedia + Gigaword) — not vectors
trained on our own 20-row sample, which would be far too small to produce meaningful geometry.

In [4]:
import gensim.downloader as api

wv = api.load("glove-wiki-gigaword-50")  # ~66MB, downloads once then caches
print(f"Loaded GloVe vectors: {len(wv.key_to_index):,} words, {wv.vector_size} dimensions\n")

# The famous analogy: king - man + woman ≈ queen
result = wv.most_similar(positive=['woman', 'king'], negative=['man'], topn=3)
print("king - man + woman ≈", result)

Loaded GloVe vectors: 400,000 words, 50 dimensions



king - man + woman ≈ [('queen', 0.8523604273796082), ('throne', 0.7664334177970886), ('prince', 0.7592144012451172)]


In [5]:
# Nearest neighbors of words from our own domain
for word in ["good", "help", "pain", "doctor"]:
    neighbors = [w for w, _ in wv.most_similar(word, topn=5)]
    print(f"{word:8} -> {neighbors}")

good     -> ['better', 'really', 'always', 'sure', 'something']
help     -> ['helping', 'bring', 'need', 'take', 'helps']
pain     -> ['suffering', 'stress', 'pains', 'stomach', 'heart']
doctor   -> ['nurse', 'physician', 'patient', 'child', 'teacher']


> The neighbors aren't random — `doctor` sits near `nurse`, `physician`, `patient`; `pain` sits near `suffering`,
> `stress`, `stomach`. That's the whole point of an embedding: distance in the vector space tracks distance in
> meaning, something TF-IDF's word-frequency counts have no way to represent.

### TF-IDF vs. Word Embeddings

| | TF-IDF | Word Embeddings |
|---|---|---|
| Represents | Word importance by frequency | Word meaning in vector space |
| Captures meaning? | No | Yes — similar words are close |
| Order / context? | No | Partially (contextual embeddings: yes) |
| Best for | Strong, fast baseline | Semantic tasks, deep learning input |

### 2.4 &nbsp;Contextual Embeddings

Word2Vec and GloVe give each word one *fixed* vector — but words are ambiguous ("bank" as in river vs. money) and a
fixed vector can't tell them apart. Contextual embeddings from transformers (Week 7: BERT) produce a *different*
vector for the same word depending on its sentence, which is why transformer-based models outperform older
approaches on nuanced language tasks.

## 🧪 Hands-On Lab: Vectorizing Text

| Step | Task | Status |
|---|---|---|
| 1 | Apply TF-IDF to Day 1's cleaned text and train a baseline classifier | 👇 below |
| 2 | Load pre-trained embeddings and inspect semantic geometry | ✅ done above (§2.3) |
| 3 | Compare TF-IDF against Week 7's LSTM/transformer on the same metric | 👇 below |
| 4 | Document which representation fits the project, and why | 👇 below |

> [!NOTE]
> **Adapting the lab to this data:** the curriculum step calls for "a simple classifier as a text baseline." Our
> capstone (Cardiac Monitoring) has no text target to classify, so this step uses the sentiment label bundled with
> `day2_sample_reviews.csv` instead — giving TF-IDF an actual prediction task to solve, on the same reviews used all
> week.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, df["sentiment"], test_size=0.3, random_state=42, stratify=df["sentiment"]
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)

print("Test accuracy:", round(accuracy_score(y_test, preds), 2), f" (n_test={len(y_test)})")
print("\nNote: with only 20 reviews total, this accuracy is illustrative of the *mechanism*, not a real evaluation —")
print("Day 5's lesson on evaluating against a proper baseline applies here just as much as to the capstone model.")

Test accuracy: 0.5  (n_test=6)

Note: with only 20 reviews total, this accuracy is illustrative of the *mechanism*, not a real evaluation —
Day 5's lesson on evaluating against a proper baseline applies here just as much as to the capstone model.


### Step 3 — TF-IDF vs. Week 7's sequence models

Week 7 built an LSTM and, separately, compared a fine-tuned DistilBERT against it for this same Drug Review text.
Re-running those models isn't practical inside this notebook (they live in `week7/day3` and `week7/day4`), so this
step is answered directly instead of re-executed:

- **TF-IDF + Logistic Regression** (today): fast, transparent, no GPU needed, but blind to word order and meaning —
  "did not help" and "did help" would only differ because "not" survived Day 1's fix, not because the model
  understands negation.
- **LSTM / DistilBERT** (Week 7): capture order and context directly, so a phrase like "did not help" is understood
  as a unit — at the cost of far more compute and much less interpretability.
- **For this capstone specifically:** none of this matters operationally — Cardiac Monitoring is tabular, not text.
  The value of Day 2 is the *skill*, not a component that ships in the final pipeline.

### 📝 Step 4 — Documentation: which representation fits, and why

- **TF-IDF** is the right default whenever the task is short on data, needs to run fast, and needs to stay
  interpretable (as just shown: it can name the *exact terms* driving a prediction).
- **Word embeddings** earn their cost when meaning and semantic similarity matter more than surface word overlap —
  e.g. matching a symptom description to a differently-worded but similar one.
- **Contextual embeddings (BERT-style)** are the right choice only when a word's meaning genuinely shifts by context
  and the project can afford the compute — which is exactly why Week 7 reached for DistilBERT on drug reviews, not
  plain Word2Vec.
- **For the Cardiac Monitoring capstone:** none of the three apply directly — the data is structured vitals, not
  text — but if the project ever ingested physician notes, TF-IDF would be the sane first baseline, same as Day 1's
  NegEx point: start simple, prove the value, then justify the added complexity of embeddings.

<div style="display:flex; flex-wrap:wrap; gap:8px; margin:18px 0;">
<span style="background:#0B2E59; color:#EAF6FA; padding:6px 14px; border-radius:8px; font-size:12px; font-weight:700;">📐 Scikit-learn (TfidfVectorizer)</span>
<span style="background:#0B2E59; color:#EAF6FA; padding:6px 14px; border-radius:8px; font-size:12px; font-weight:700;">🧠 Gensim / pre-trained embeddings</span>
<span style="background:#0B2E59; color:#EAF6FA; padding:6px 14px; border-radius:8px; font-size:12px; font-weight:700;">📓 Jupyter / Colab</span>
<span style="background:#0B2E59; color:#EAF6FA; padding:6px 14px; border-radius:8px; font-size:12px; font-weight:700;">🔗 Git &amp; GitHub</span>
</div>

<div style="background:linear-gradient(135deg,#0B2E59,#061A33); border-radius:14px; padding:24px 28px; color:#EAF6FA; margin-top:10px;">
<h3 style="margin-top:0; color:#8CE9F4;">✅ Day 2 — Closed Out</h3>
<p style="line-height:1.6;">
Clean tokens are now numbers two different ways — sparse-and-interpretable (TF-IDF) and dense-and-semantic
(embeddings) — with a documented, justified choice between them. Tomorrow (<b>Day 3</b>) the sprint switches modality
entirely: computer vision preprocessing with OpenCV, matching images to what a pre-trained model expects.
</p>
<div style="text-align:right; font-size:13px; opacity:0.7; margin-top:10px;">Week 8 · Sprint 3 · Day 2 of 5 → <b>Day 3: Computer Vision Preprocessing</b></div>
</div>